# Hypothesis Tests: t-tests and Chi-squared Tests

The explainer posed a question that comes up constantly in analytical work: "Complaints are down 12%. Did the change work?" The honest answer is that the data alone cannot tell us. What it *can* tell us is how likely the observed result would be if nothing had actually changed. That reasoning is hypothesis testing.

In this notebook we will apply that framework to the clinical trial. The regulatory body needs to know whether the drug works, and a glance at group averages is not enough. We need a method that accounts for the sampling variability we explored in the previous notebook and quantifies the strength of the evidence.

By the end of this notebook we will be able to:

- Formulate **null and alternative hypotheses** for a clinical question
- Run a two-sample **t-test** with `scipy.stats.ttest_ind`
- Interpret a **p-value** correctly (and avoid common misinterpretations)
- Run a **chi-squared test** with `scipy.stats.chi2_contingency`
- Calculate **Cohen's d** as a measure of effect size
- Distinguish between statistical significance and practical significance

In [ ]:
%pip install -q pandas matplotlib seaborn scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
np.random.seed(42)

In [ ]:
patients = pd.read_csv("../data/trial_patients.csv")
outcomes = pd.read_csv("../data/treatment_outcomes.csv")
adverse = pd.read_csv("../data/adverse_events.csv")

df = patients.merge(outcomes, on="patient_id")
print(f"Patients: {len(df)}")
df.head()

---

## 1. Exploring the groups

Before running any tests, look at the data. We know from the previous notebook that sample statistics vary, so start by getting a feel for both groups. Summary statistics and plots first; formal tests after.

In [ ]:
df.groupby("group")[["baseline_score", "week_12_score"]].describe().round(2)

In [ ]:
treatment = df[df["group"] == "treatment"]
control = df[df["group"] == "control"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(treatment["week_12_score"], bins=30, alpha=0.6, label="Treatment", edgecolor="white")
axes[0].hist(control["week_12_score"], bins=30, alpha=0.6, label="Control", edgecolor="white")
axes[0].set_xlabel("Week 12 score")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Week 12 scores by group")
axes[0].legend()

# Score change from baseline
df["score_change"] = df["week_12_score"] - df["baseline_score"]
treatment = df[df["group"] == "treatment"]
control = df[df["group"] == "control"]

axes[1].hist(treatment["score_change"], bins=30, alpha=0.6, label="Treatment", edgecolor="white")
axes[1].hist(control["score_change"], bins=30, alpha=0.6, label="Control", edgecolor="white")
axes[1].set_xlabel("Score change (week 12 - baseline)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Score change by group")
axes[1].legend()

plt.tight_layout()
plt.show()

The treatment group's scores shift lower (improved) compared to the control group. We can see the difference in the histograms. But remember what the sampling distribution showed us: sample means bounce around. Is this difference large enough that it could not plausibly be explained by that bounce?

---

## 2. Formulating hypotheses

The explainer introduced the logic: every hypothesis test starts with two competing explanations, and the null hypothesis always gets the benefit of the doubt. Think of it like a court trial. The drug is presumed to have no effect (innocent) until the evidence is strong enough to conclude otherwise.

For this clinical trial:

- **Null hypothesis** ($H_0$): There is no difference between the treatment and control groups. Any observed difference is due to random sampling. ($\mu_T = \mu_C$)
- **Alternative hypothesis** ($H_1$): There is a real difference between the groups. ($\mu_T \neq \mu_C$)

The test calculates how likely we would be to see a difference as large as the one observed, *assuming $H_0$ is true*.

---

## 3. Two-sample t-test

Time to let the data speak. The **two-sample t-test** compares the means of two independent groups, accounting for the variability within each group and the size of the samples. The function `scipy.stats.ttest_ind` returns a t-statistic and a p-value.

In [ ]:
treatment_scores = treatment["week_12_score"].values
control_scores = control["week_12_score"].values

t_stat, p_value = stats.ttest_ind(treatment_scores, control_scores)

print(f"Treatment mean: {treatment_scores.mean():.2f}")
print(f"Control mean:   {control_scores.mean():.2f}")
print(f"Difference:     {treatment_scores.mean() - control_scores.mean():.2f}")
print(f"t-statistic:    {t_stat:.4f}")
print(f"p-value:        {p_value:.6f}")

### Interpreting the p-value

This is the number the explainer spent the most time on, and for good reason. The **p-value** is the probability of observing a result as extreme as (or more extreme than) the one obtained, *if the null hypothesis were true*. It measures how surprising the data is under the assumption of no effect.

The conventional threshold is $\alpha = 0.05$. If $p < 0.05$, we reject the null hypothesis and call the result **statistically significant**.

But recall what the p-value does **not** mean:

- It is not the probability that $H_0$ is true.
- It is not the probability that the result happened by chance.
- It does not tell us the size of the effect.

The American Statistical Association issued a formal statement correcting these misinterpretations. Keep them in mind as we work through what follows.

In [ ]:
alpha = 0.05

if p_value < alpha:
    print(f"p = {p_value:.6f} < {alpha}")
    print("Reject the null hypothesis: statistically significant difference between groups.")
else:
    print(f"p = {p_value:.6f} >= {alpha}")
    print("Fail to reject the null hypothesis: no statistically significant difference detected.")

---

## 4. Checking baseline balance

A well-designed trial should have similar baseline characteristics across groups. If the groups differed before treatment began, any difference at week 12 might reflect pre-existing differences rather than the drug. This is a sanity check on the trial's randomisation.

In [ ]:
t_base, p_base = stats.ttest_ind(
    treatment["baseline_score"].values,
    control["baseline_score"].values,
)

print(f"Baseline t-statistic: {t_base:.4f}")
print(f"Baseline p-value:     {p_base:.4f}")
print()
if p_base >= 0.05:
    print("No significant difference at baseline. The groups started on equal footing.")
else:
    print("Significant baseline difference detected. Consider adjusting for this in the analysis.")

---

## 5. One-sample t-test

Sometimes we want to test whether a single group's mean differs from a specific value, rather than comparing two groups. For instance: did the treatment group's symptoms actually change from baseline, or did they stay flat?

In [ ]:
treatment_change = treatment["score_change"].values

t_one, p_one = stats.ttest_1samp(treatment_change, popmean=0)

print(f"Mean score change (treatment): {treatment_change.mean():.2f}")
print(f"t-statistic: {t_one:.4f}")
print(f"p-value:     {p_one:.6f}")
print()
if p_one < 0.05:
    print("The treatment group's score change is significantly different from zero.")
else:
    print("Cannot conclude the treatment group's score changed from zero.")

---

## 6. Chi-squared test for outcome proportions

The t-test works with continuous variables like scores. But the trial also records categorical outcomes: did each patient improve, stay stable, or worsen? For categorical data, the appropriate tool is the **chi-squared test of independence**, which the explainer introduced with the hospital triage example. It compares observed counts to the counts we would expect if the two variables (group and outcome) were unrelated.

In [ ]:
contingency = pd.crosstab(df["group"], df["final_outcome"])
print(contingency)
print()

# Show proportions
proportions = pd.crosstab(df["group"], df["final_outcome"], normalize="index").round(3)
print(proportions)

In [ ]:
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)

print(f"Chi-squared statistic: {chi2:.4f}")
print(f"p-value:               {p_chi:.6f}")
print(f"Degrees of freedom:    {dof}")
print()
if p_chi < 0.05:
    print("The outcome distribution differs significantly between groups.")
else:
    print("No significant difference in outcome distribution between groups.")

In [ ]:
# Visualise the outcome proportions
proportions.plot(kind="bar", figsize=(8, 4), edgecolor="white")
plt.ylabel("Proportion")
plt.title("Outcome proportions by group")
plt.xticks(rotation=0)
plt.legend(title="Outcome")
plt.tight_layout()
plt.show()

---

## 7. Effect size: Cohen's d

Here is where the explainer's distinction between statistical significance and practical significance becomes concrete. A p-value tells us whether an effect exists, but not how large it is. With enough patients, even a trivially small improvement becomes statistically significant. (Remember the headache drug example: 12 minutes' improvement, p = 0.03, 5,000 patients.)

**Cohen's d** measures the magnitude of the difference in standardised units. It is the difference in means divided by the pooled standard deviation:

$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_p}$$

where the pooled standard deviation is:

$$s_p = \sqrt{\frac{(n_1 - 1)s_1^2 + (n_2 - 1)s_2^2}{n_1 + n_2 - 2}}$$

Rules of thumb (Cohen, 1988):

| d | Interpretation |
|---|----------------|
| 0.2 | Small |
| 0.5 | Medium |
| 0.8 | Large |

In [ ]:
def cohens_d(group1, group2):
    """Calculate Cohen's d for two independent groups."""
    n1, n2 = len(group1), len(group2)
    s1, s2 = group1.std(ddof=1), group2.std(ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_std


d = cohens_d(treatment["week_12_score"].values, control["week_12_score"].values)
print(f"Cohen's d: {d:.3f}")
print()

if abs(d) < 0.2:
    print("Negligible effect size.")
elif abs(d) < 0.5:
    print("Small effect size.")
elif abs(d) < 0.8:
    print("Medium effect size.")
else:
    print("Large effect size.")

---

## 8. Statistical significance vs practical significance

We now have both numbers in front of us: the p-value and the effect size. A result can be statistically significant (low p-value) but practically unimportant (small d). The reverse is also possible: a meaningful effect can fail to reach significance in a small sample because the study was underpowered.

Good practice is to report **both** together:

- The p-value (evidence against the null hypothesis)
- The effect size (magnitude of the difference)
- Confidence intervals (covered in the next notebook)

In [ ]:
print("=" * 50)
print("TRIAL RESULTS SUMMARY")
print("=" * 50)
print(f"Treatment group mean (week 12): {treatment_scores.mean():.2f}")
print(f"Control group mean (week 12):   {control_scores.mean():.2f}")
print(f"Mean difference:                {treatment_scores.mean() - control_scores.mean():.2f}")
print(f"t-statistic:                    {t_stat:.4f}")
print(f"p-value:                        {p_value:.6f}")
print(f"Cohen's d:                      {d:.3f}")
print("=" * 50)

---

## 9. Exercises

We have seen the core workflow: formulate hypotheses, run a test, interpret the p-value, and check the effect size. These exercises ask you to apply that workflow to different slices of the trial data and to grapple with a problem that arises whenever you run multiple tests.

### Exercise 1: t-test on score change

Instead of comparing raw week 12 scores, compare the **score change** (week 12 minus baseline) between treatment and control groups using a two-sample t-test. Report the t-statistic, p-value, and Cohen's d. Does the conclusion change?

In [ ]:
# Your code here


### Exercise 2: Chi-squared test for adverse events

The `adverse_events.csv` file records adverse events by patient. Merge it with `trial_patients.csv` to get each patient's group. Build a contingency table of `group` vs `event_type` and run a chi-squared test. Are adverse event types distributed differently between treatment and control?

In [ ]:
# Your code here


### Exercise 3: Site-level variation

The trial ran across 8 sites. For each site, run a two-sample t-test comparing treatment vs control `week_12_score`. Store the results in a DataFrame with columns `site_id`, `t_stat`, `p_value`, and `cohens_d`. Do all sites show the same direction of effect? Are all sites individually significant at $\alpha = 0.05$?

In [ ]:
# Your code here


### Exercise 4: Multiple testing

In Exercise 3 you ran 8 t-tests. When you run multiple tests, each with $\alpha = 0.05$, the chance of at least one false positive increases. Apply the **Bonferroni correction**: divide $\alpha$ by the number of tests ($0.05 / 8 = 0.00625$). Re-evaluate which sites are significant under this stricter threshold. How many remain significant?

In [ ]:
# Your code here


---

## Summary

In this notebook we:

- Formulated **null and alternative hypotheses** for the clinical trial
- Ran a **two-sample t-test** to compare week 12 scores between treatment and control
- Interpreted the **p-value** as the probability of the observed result under $H_0$
- Checked baseline balance with a t-test to verify the trial design
- Used a **one-sample t-test** to test whether score change differed from zero
- Applied the **chi-squared test** to compare categorical outcome proportions
- Calculated **Cohen's d** to quantify effect size independent of sample size
- Recognised that statistical significance and practical significance are separate questions

The p-value told us whether the drug's effect is likely real. But "real" is not the same as "useful." A regulator needs to know *how much* the drug helps, not just that it helps at all. In the next notebook, we will build **confidence intervals** to estimate the range of plausible effect sizes and learn how to communicate that uncertainty to a non-technical audience.